This script presents first 10 queries to be used as starting sample to generate more of them.
The queries present three different levels of complexity. They have been transformed to questions in natural language.
Such transformation process is firstly done by Gemini. Later other possibilities/models from literature are explored. The aim is to get to automatize such process.
The correctness of the queries has been checked through astroquery.gaia library (using asyncr).

In [3]:
# ── SSL fix for ARNES cluster ─────────────────────────────────────────────────
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [2]:
%pip install astroquery

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 49.0 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [astroquery]3 [astroquery]
Note: you may need to restart the kernel to use updated packages.


In [4]:
queries = [
    {
        "id": 1,
        "complexity": "Simple",
        "sql": """
SELECT ra, dec, phot_g_mean_mag 
FROM gaiadr3.gaia_source 
WHERE phot_g_mean_mag < 10 
AND random_index < 1000000
        """
    },
    {
        "id": 2,
        "complexity": "Simple",
        "sql": """
SELECT ra, dec, pmra, pmdec 
FROM gaiadr3.gaia_source 
WHERE (pmra*pmra + pmdec*pmdec) > 10000 
AND random_index < 1000000
        """
    },
    {
        "id": 3,
        "complexity": "Medium",
        "sql": """
SELECT (phot_bp_mean_mag - phot_rp_mean_mag) AS bp_rp, phot_g_mean_mag 
FROM gaiadr3.gaia_source 
WHERE phot_bp_mean_mag IS NOT NULL 
AND phot_rp_mean_mag IS NOT NULL 
AND random_index < 1000000
        """
    },
    {
        "id": 4,
        "complexity": "Medium",
        "sql": """
SELECT ra, dec, parallax 
FROM gaiadr3.gaia_source 
WHERE CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 56.75, 24.12, 1.0)) = 1 
AND random_index < 1000000
        """
    },
    {
        "id": 5,
        "complexity": "Medium",
        "sql": """
SELECT l, b, phot_g_mean_mag 
FROM gaiadr3.gaia_source 
WHERE b BETWEEN -1 AND 1 
AND random_index < 1000000
        """
    },
    {
        "id": 6,
        "complexity": "Medium",
        "sql": """
SELECT ra, dec, parallax, radial_velocity 
FROM gaiadr3.gaia_source 
WHERE parallax > 100 
AND radial_velocity IS NOT NULL 
AND random_index < 1000000
        """
    },
    {
        "id": 7,
        "complexity": "Complex",
        "sql": """
SELECT ra, dec, parallax, pmra, pmdec, radial_velocity 
FROM gaiadr3.gaia_source 
WHERE parallax > 10 
AND radial_velocity IS NOT NULL 
AND random_index < 1000000
        """
    },
    {
        "id": 8,
        "complexity": "Complex",
        "sql": """
SELECT ra, dec, phot_g_mean_mag, ag_gspphot 
FROM gaiadr3.gaia_source 
WHERE CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 266.41, -29.00, 0.5)) = 1 
AND random_index < 1000000
        """
    },
    {
        "id": 9,
        "complexity": "Complex",
        "sql": """
SELECT g.source_id, g.ra, g.dec, d.dist_50 
FROM gaiadr3.gaia_source AS g 
JOIN external.gaiaedr3_distance AS d ON g.source_id = d.source_id 
WHERE g.random_index < 1000000 
AND d.dist_50 < 500
        """
    },
    {
        "id": 10,
        "complexity": "Complex",
        "sql": """
SELECT l, b, ag_gspphot, distance_gspphot 
FROM gaiadr3.gaia_source 
WHERE ag_gspphot > 0.5 
AND distance_gspphot < 1000 
AND random_index < 1000000
        """
    }
]

In [5]:
import time
from astroquery.gaia import Gaia

def validate_queries(query_list):
    jobs = []
    
    print(f"Submitting {len(query_list)} queries for validation...")
    
    # 1. Submit all jobs asynchronously
    for item in query_list:
        print(f"Launching Query {item['id']} ({item['complexity']})...")
        #job = Gaia.launch_job_async(item['sql'])
        # Try this for one query to see if the server responds
        job = Gaia.launch_job(queries[0]['sql'])
        jobs.append({
            "id": item['id'],
            "job_obj": job
        })

    # 2. Monitor status
    print("\nMonitoring job status...")
    completed = False
    while not completed:
        statuses = [j['job_obj'].get_phase() for j in jobs]
        
        # Check if all jobs are in a terminal state (COMPLETED, ERROR, or ABORTED)
        if all(phase in ['COMPLETED', 'ERROR', 'ABORTED'] for phase in statuses):
            completed = True
        else:
            print(f"Current Statuses: {statuses}")
            time.sleep(2) # Wait 2 seconds before checking again

    # 3. Report Results
    print("\n--- Validation Results ---")
    for j in jobs:
        phase = j['job_obj'].get_phase()
        if phase == 'COMPLETED':
            # We fetch just 1 row to prove it's computable without downloading everything
            res = j['job_obj'].get_results()
            print(f"Query {j['id']}: SUCCESS (Rows found: {len(res)})")
        else:
            print(f"Query {j['id']}: FAILED (Status: {phase})")

# Run the validation
validate_queries(queries)

In preparation for Gaia DR4, the Gaia archive is in evolution. Unfortunately, it may be unstable at times and particular types of queries may time out. Please consider registering for a user account (https://www.cosmos.esa.int/web/gaia-users/register). For questions or advice, please contact the Gaia helpdesk (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk).
Submitting 10 queries for validation...
Launching Query 1 (Simple)...
Launching Query 2 (Simple)...
Launching Query 3 (Medium)...
Launching Query 4 (Medium)...
Launching Query 5 (Medium)...
Launching Query 6 (Medium)...
Launching Query 7 (Complex)...
Launching Query 8 (Complex)...
Launching Query 9 (Complex)...
Launching Query 10 (Complex)...

Monitoring job status...

--- Validation Results ---
Query 1: SUCCESS (Rows found: 277)
Query 2: SUCCESS (Rows found: 277)
Query 3: SUCCESS (Rows found: 277)
Query 4: SUCCESS (Rows found: 277)
Query 5: SUCCESS (Rows found: 277)
Query 6: SUCCESS (Rows found: 277)
Query 7: SUCCESS (Rows foun

In [ ]:
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# --- RE-DEFINE DATAFRAME (In case it was lost on restart) ---
data = [
    {
        "id": 1,
        "complexity": "Simple",
        "question": "Show the equatorial coordinates and G-band magnitude for stars brighter than magnitude 10.",
        "sql": "SELECT ra, dec, phot_g_mean_mag FROM gaiadr3.gaia_source WHERE phot_g_mean_mag < 10 AND random_index < 1000000"
    },
    {
        "id": 2,
        "complexity": "Simple",
        "question": "Find stars with a total proper motion exceeding 100 mas/yr.",
        "sql": "SELECT ra, dec, pmra, pmdec FROM gaiadr3.gaia_source WHERE (pmra*pmra + pmdec*pmdec) > 10000 AND random_index < 1000000"
    },
    {
        "id": 3,
        "complexity": "Medium",
        "question": "Calculate the BP-RP color and return it along with the G-band magnitude for a sample of stars.",
        "sql": "SELECT (phot_bp_mean_mag - phot_rp_mean_mag) AS bp_rp, phot_g_mean_mag FROM gaiadr3.gaia_source WHERE phot_bp_mean_mag IS NOT NULL AND phot_rp_mean_mag IS NOT NULL AND random_index < 1000000"
    },
    {
        "id": 4,
        "complexity": "Medium",
        "question": "Retrieve the positions and parallaxes of stars within a 1-degree radius of the Pleiades cluster.",
        "sql": "SELECT ra, dec, parallax FROM gaiadr3.gaia_source WHERE CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 56.75, 24.12, 1.0)) = 1 AND random_index < 1000000"
    },
    {
        "id": 5,
        "complexity": "Medium",
        "question": "Find stars located within 1 degree of the Galactic plane and show their Galactic coordinates.",
        "sql": "SELECT l, b, phot_g_mean_mag FROM gaiadr3.gaia_source WHERE b BETWEEN -1 AND 1 AND random_index < 1000000"
    },
    {
        "id": 6,
        "complexity": "Medium",
        "question": "List stars with a parallax greater than 100 mas that also have available radial velocity measurements.",
        "sql": "SELECT ra, dec, parallax, radial_velocity FROM gaiadr3.gaia_source WHERE parallax > 100 AND radial_velocity IS NOT NULL AND random_index < 1000000"
    },
    {
        "id": 7,
        "complexity": "Complex",
        "question": "Retrieve full 6D phase-space data for stars within approximately 100 parsecs of the Sun.",
        "sql": "SELECT ra, dec, parallax, pmra, pmdec, radial_velocity FROM gaiadr3.gaia_source WHERE parallax > 10 AND radial_velocity IS NOT NULL AND random_index < 1000000"
    },
    {
        "id": 8,
        "complexity": "Complex",
        "question": "Identify stars within 0.5 degrees of the Galactic Center and provide their G-band extinction values.",
        "sql": "SELECT ra, dec, phot_g_mean_mag, ag_gspphot FROM gaiadr3.gaia_source WHERE CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 266.41, -29.00, 0.5)) = 1 AND random_index < 1000000"
    },
    {
        "id": 9,
        "complexity": "Complex",
        "question": "Join the main source table with the distance catalog to find stars within 500 parsecs.",
        "sql": "SELECT g.source_id, g.ra, g.dec, d.dist_50 FROM gaiadr3.gaia_source AS g JOIN external.gaiaedr3_distance AS d ON g.source_id = d.source_id WHERE g.random_index < 1000000 AND d.dist_50 < 500"
    },
    {
        "id": 10,
        "complexity": "Complex",
        "question": "Find stars with high interstellar absorption and return their estimated distances.",
        "sql": "SELECT l, b, ag_gspphot, distance_gspphot FROM gaiadr3.gaia_source WHERE ag_gspphot > 0.5 AND distance_gspphot < 1000 AND random_index < 1000000"
    }
]

df_nlp = pd.DataFrame(data)
# To view the structure
print(df_nlp.head())

In [6]:
%pip install transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 60.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.0/888.0 MB 57.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 67.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 79.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 60.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 79.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.4 MB/s et

In [7]:
%pip install SentencePiece 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [8]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# --- LOAD MODEL ---
model_name = "mrm8488/t5-base-finetuned-wikiSQL"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def automate_sql_to_question(sql_query):
    input_text = f"translate sql to english: {sql_query}"
    inputs = tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(inputs, max_length=128, num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# --- TEST ---
print("--- New Automated Translation Test ---")
test_sql = df_nlp.iloc[0]['sql']
print(f"Result: {automate_sql_to_question(test_sql)}")

/d/hpc/home/ps92930/.conda/envs/bd39/lib/python3.9/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/d/hpc/home/ps92930/.conda/envs/bd39/lib/python3.9/site-packages/huggingface_hub/file_download.py:805: UserWarning: Not enough free disk space to download the file. The expected file size is: 1187.80 MB. The target location /scratch/hf_cache/models--mrm8488--t5-base-finetuned-wikiSQL/blobs only has 15.91 MB free disk space.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

OSError: [Errno 28] No space left on device